In [70]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_samples, silhouette_score
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder

In [71]:
df1 = pd.read_csv('/content/titles.csv')
df2 = pd.read_csv('/content/titles1.csv')
df3 = pd.read_csv('/content/titles2.csv')

In [72]:
df = pd.concat([df1, df2, df3])
df.head()

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
0,ts300399,Five Came Back: The Reference Films,SHOW,This collection includes 12 World War II-era p...,1945,TV-MA,51,['documentation'],['US'],1.0,NaN,NaN,NaN,0.600,NaN
1,tm84618,Taxi Driver,MOVIE,A mentally unstable Vietnam War veteran works ...,1976,R,114,"['drama', 'crime']",['US'],NaN,tt0075314,8.2,808582.0,40.965,8.179
2,tm154986,Deliverance,MOVIE,Intent on seeing the Cahulawassee River before...,1972,R,109,"['drama', 'action', 'thriller', 'european']",['US'],NaN,tt0068473,7.7,107673.0,10.010,7.300
3,tm127384,Monty Python and the Holy Grail,MOVIE,"King Arthur, accompanied by his squire, recrui...",1975,PG,91,"['fantasy', 'action', 'comedy']",['GB'],NaN,tt0071853,8.2,534486.0,15.461,7.811
4,tm120801,The Dirty Dozen,MOVIE,12 American military prisoners in World War II...,1967,NaN,150,"['war', 'action']","['GB', 'US']",NaN,tt0061578,7.7,72662.0,20.398,7.600


**Data preprocessing**

In [73]:
df.drop_duplicates()

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
0,ts300399,Five Came Back: The Reference Films,SHOW,This collection includes 12 World War II-era p...,1945,TV-MA,51,['documentation'],['US'],1.0,NaN,NaN,NaN,0.600,NaN
1,tm84618,Taxi Driver,MOVIE,A mentally unstable Vietnam War veteran works ...,1976,R,114,"['drama', 'crime']",['US'],NaN,tt0075314,8.2,808582.0,40.965,8.179
2,tm154986,Deliverance,MOVIE,Intent on seeing the Cahulawassee River before...,1972,R,109,"['drama', 'action', 'thriller', 'european']",['US'],NaN,tt0068473,7.7,107673.0,10.010,7.300
3,tm127384,Monty Python and the Holy Grail,MOVIE,"King Arthur, accompanied by his squire, recrui...",1975,PG,91,"['fantasy', 'action', 'comedy']",['GB'],NaN,tt0071853,8.2,534486.0,15.461,7.811
4,tm120801,The Dirty Dozen,MOVIE,12 American military prisoners in World War II...,1967,NaN,150,"['war', 'action']","['GB', 'US']",NaN,tt0061578,7.7,72662.0,20.398,7.600
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9866,tm510327,Lily Is Here,MOVIE,Dallas and heroin have one thing in common: Du...,2021,NaN,93,['drama'],['US'],NaN,tt7672388,5.3,20.0,1.406,NaN
9867,tm1079144,Jay Nog: Something from Nothing,MOVIE,Something From Nothing takes you on a stand-up...,2021,NaN,55,['comedy'],['US'],NaN,tt15041600,NaN,NaN,0.600,NaN
9868,tm847725,Chasing,MOVIE,A cop from Chennai sets out to nab a dreaded d...,2021,NaN,116,['crime'],['IN'],NaN,NaN,NaN,NaN,1.960,NaN
9869,tm1054116,Baikunth,MOVIE,"This story is about prevalent caste problem, e...",2021,NaN,72,"['family', 'drama']",[],NaN,tt14331982,8.4,49.0,0.645,NaN


In [74]:
df.drop(['description', 'age_certification'], axis=1, inplace=True)

In [75]:
df['production_countries']

,production_countries
0,['US']
1,['US']
2,['US']
3,['GB']
4,"['GB', 'US']"
...,...
9866,['US']
9867,['US']
9868,['IN']
9869,[]


In [76]:
df['production_countries'] = df['production_countries'].str.replace(r"\[", '', regex=True). str.replace(']', '', regex=True).str.replace("'", '', regex=True).str.replace("\]", '', regex=True)
df['lead_prod_countries'] = df['production_countries'].str.split(',').str[0]
df['prod_countries_cnt'] = df['production_countries'].str.split(',').str.len()
df['lead_prod_countries'] = df['lead_prod_countries'].replace('', np.nan)
df.head()

,id,title,type,release_year,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score,lead_prod_countries,prod_countries_cnt
0,ts300399,Five Came Back: The Reference Films,SHOW,1945,51,['documentation'],US,1.0,NaN,NaN,NaN,0.600,NaN,US,1
1,tm84618,Taxi Driver,MOVIE,1976,114,"['drama', 'crime']",US,NaN,tt0075314,8.2,808582.0,40.965,8.179,US,1
2,tm154986,Deliverance,MOVIE,1972,109,"['drama', 'action', 'thriller', 'european']",US,NaN,tt0068473,7.7,107673.0,10.010,7.300,US,1
3,tm127384,Monty Python and the Holy Grail,MOVIE,1975,91,"['fantasy', 'action', 'comedy']",GB,NaN,tt0071853,8.2,534486.0,15.461,7.811,GB,1
4,tm120801,The Dirty Dozen,MOVIE,1967,150,"['war', 'action']","GB, US",NaN,tt0061578,7.7,72662.0,20.398,7.600,GB,2


In [ ]:
df['lead_prod_countries']

In [78]:
df['genres']

,genres
0,['documentation']
1,"['drama', 'crime']"
2,"['drama', 'action', 'thriller', 'european']"
3,"['fantasy', 'action', 'comedy']"
4,"['war', 'action']"
...,...
9866,['drama']
9867,['comedy']
9868,['crime']
9869,"['family', 'drama']"


In [79]:
df['genres'] = df['genres'].str.replace(r"\[", '', regex=True). str.replace("'", '', regex=True). str.replace('\]', '', regex=True)
df['main_genre'] = df['genres']. str.split(',').str[0]
df['main_genre'] = df['main_genre'].replace('', np.nan)

df['main_genre'].head()

,main_genre
0,documentation
1,drama
2,drama
3,fantasy
4,war


In [80]:
df.drop(['genres', 'production_countries'], axis=1, inplace=True)

In [81]:
df.shape

(19015, 14)

In [82]:
df.isnull().sum()

,0
id,0
title,1
type,0
release_year,0
runtime,0
seasons,14796
imdb_id,1396
imdb_score,1875
imdb_votes,1912
tmdb_popularity,671


In [83]:
df.dropna(inplace=True)
df.set_index('title', inplace=True)
df.drop(['id', 'imdb_id'], axis=1, inplace=True)

In [84]:
df.shape

(3303, 11)

**encoding catagorical data**

In [85]:
dummies = pd.get_dummies(df[['type', 'lead_prod_countries', 'main_genre']], drop_first=True)
df_dum = pd.concat([df, dummies], axis=1)
df_dum.drop(['type', 'lead_prod_countries', 'main_genre',], axis=1, inplace=True)

In [86]:
df_dum.dtypes

,0
release_year,int64
runtime,int64
seasons,float64
imdb_score,float64
imdb_votes,float64
...,...
main_genre_scifi,bool
main_genre_sport,bool
main_genre_thriller,bool
main_genre_war,bool


In [87]:
scaler = MinMaxScaler()
df_scaled = scaler.fit_transform(df_dum)
df_scaled = pd.DataFrame(df_scaled, columns=df_dum.columns)
df_scaled.head()

,release_year,runtime,seasons,imdb_score,imdb_votes,tmdb_popularity,tmdb_score,prod_countries_cnt,lead_prod_countries_AR,lead_prod_countries_AT,...,main_genre_history,main_genre_horror,main_genre_music,main_genre_reality,main_genre_romance,main_genre_scifi,main_genre_sport,main_genre_thriller,main_genre_war,main_genre_western
0,0.397727,0.168539,0.058824,0.9125,0.037009,0.007913,0.815870,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.625000,0.134831,0.156863,0.9250,0.155671,0.058490,0.815326,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.545455,0.286517,0.058824,0.6750,0.017194,0.022579,0.728261,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
3,0.568182,0.056180,0.450980,0.6250,0.002570,0.018954,0.619565,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.625000,0.129213,0.078431,0.7000,0.017658,0.008919,0.782609,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


DBSCAN MODEL

In [88]:
#find the esp and min_sample
eps_array = [0.2, 0.5, 1]
min_samples_array = [5, 10,30]

for eps in eps_array:
  for min_samples in min_samples_array:
    dbscan = DBSCAN(eps=eps, min_samples=min_samples).fit(df_scaled)
    #retrive cluster
    labels = dbscan.labels_
    if len(set(labels)) == 1:
      continue
    silhouette_avg = silhouette_score(df_scaled, labels)
    print("for eps = ", eps,
          "for min_samples = ", min_samples,
          "count clusters = ", len(set(labels)),
          "The average silhouette_score : ", silhouette_avg
          )

for eps =  0.2 for min_samples =  5 count clusters =  75 The average silhouette_score :  0.4378646549782871
for eps =  0.2 for min_samples =  10 count clusters =  37 The average silhouette_score :  0.36654867954788023
for eps =  0.2 for min_samples =  30 count clusters =  17 The average silhouette_score :  0.23131488163210734
for eps =  0.5 for min_samples =  5 count clusters =  91 The average silhouette_score :  0.6021028769243768
for eps =  0.5 for min_samples =  10 count clusters =  56 The average silhouette_score :  0.5307158038680273
for eps =  0.5 for min_samples =  30 count clusters =  21 The average silhouette_score :  0.3621888505171278
for eps =  1 for min_samples =  5 count clusters =  93 The average silhouette_score :  0.6092943039744223
for eps =  1 for min_samples =  10 count clusters =  57 The average silhouette_score :  0.5366122080935836
for eps =  1 for min_samples =  30 count clusters =  22 The average silhouette_score :  0.3710964146530399


**DBSCAN With Best Hypterparameters (eps=1, minpnts=5)**

In [89]:
dbscan_model = DBSCAN(eps=1, min_samples =5).fit(df_scaled)
df['dbscan_cluster'] = dbscan_model.labels_
df['dbscan_cluster'].value_counts()

,count
dbscan_cluster,
-1,388
1,349
13,267
35,221
5,146
...,...
76,5
82,5
78,5


**Movie Recommendation Function**

In [90]:
import random
def recommend_movies(movie_name : str):
  movie_name = movie_name.lower()

  df['name'] = df.index.str.lower()

  movie = df[df['name'].str.contains(movie_name, na=False)]

  if not movie.empty:
    cluster = movie['dbscan_cluster'].values[0] #get the input cluster
    #get all movies in same cluster
    cluster_movies = df[df['dbscan_cluster'] == cluster]

    if len(cluster_movies) >=5:
      recommended_movies = random.sample(list(cluster_movies.index), 5)
    else:
      recommended_movies = list(cluster_movies.index)

    print("we can recommend you these movies : ")
    for m in recommended_movies:
        print(m)
  else:
    print("Movie is not found in this dataset")

**Input**

In [92]:
movie_name = input("Enter a movie name : ")
print("\n")
recommend_movies(movie_name)

Enter a movie name : hello


we can recommend you these movies : 
Hello, Me!
The Sound of Your Heart: Reboot
Flower Boy Next Door
Her Private Life
Revolutionary Love
